# ¿Qué opinan los clientes de Wise?
## Análisis de reseñas de Trustpilot — explicado sin tecnicismos

> **Para quién es este documento:** Equipos de negocio, producto y marketing que quieren
> entender qué dicen los clientes sin necesidad de saber programar.

---

### ¿Qué hemos hecho?

Hemos analizado **100 opiniones reales de clientes de Wise.com** publicadas en Trustpilot,
más otras **5.700 opiniones del sector financiero** (competidores), para responder tres preguntas:

1. **¿Los clientes están contentos o descontentos con Wise?** ¿Y comparado con la competencia?
2. **¿De qué hablan en sus reseñas?** ¿Precio, app, atención al cliente...?
3. **¿En qué temas estamos bien y en cuáles hay margen de mejora?**

---

### ¿Cómo lo hemos hecho? (versión sencilla)

Hemos usado dos programas de inteligencia artificial que leen texto y entienden su significado:

| Programa | Qué hace |
|---|---|
| **Detector de sentimiento** | Lee cada reseña y decide si es positiva, neutral o negativa |
| **Agrupador de temas** | Lee todas las reseñas y las agrupa por el tema del que hablan |

Piensa en ello como tener un analista que lee 5.800 reseñas en minutos y te da un resumen.


---
## Paso 1: Preparar el análisis

Antes de empezar a leer reseñas, el programa necesita:
- Saber **qué empresa analizar** (Wise.com)
- Saber **de qué sector** vienen sus competidores (Money & Insurance)
- Cargar las herramientas de inteligencia artificial

Es como preparar el escritorio antes de empezar a trabajar.


In [ ]:
# Cargamos las herramientas que necesitamos
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

# Definimos qué empresa y sector vamos a analizar
CSV_PATH = 'trustpilot-reviews-123k.csv'  # El archivo con todas las reseñas
TARGET   = 'wise.com'                      # Empresa que analizamos
CATEGORY = 'Money & Insurance'             # Sector de la competencia

# Esta función convierte estrellas (1-5) en palabras comprensibles
# 1-2 estrellas = cliente insatisfecho (Negativo)
# 3 estrellas = cliente indiferente (Neutro)
# 4-5 estrellas = cliente satisfecho (Positivo)
def stars_to_label(s):
    if s <= 2: return 'Negativo'
    if s == 3: return 'Neutro'
    return 'Positivo'

PALETTE = {'Positivo': '#2ecc71', 'Neutro': '#f39c12', 'Negativo': '#e74c3c'}
print('Todo listo para empezar el análisis.')


---
## Paso 2: Cargar las reseñas

Cargamos el fichero con todas las reseñas. El dataset contiene:
- **123.000 reseñas** de muchas empresas y sectores
- Cada empresa tiene **exactamente 100 reseñas**: 20 de cada puntuación (1 a 5 estrellas)
  — esto es importante: el dataset está equilibrado a propósito para no favorecer a ninguna empresa

De ese total, nos quedamos con:
- Las **100 reseñas de Wise** para analizarlas en detalle
- Las **5.729 reseñas del sector financiero** para comparar con la competencia


In [ ]:
# Cargamos el archivo con todas las reseñas
df_all = pd.read_csv(CSV_PATH)
print(f'Total de reseñas en el dataset: {df_all.shape[0]:,}')
print(f'Columnas disponibles: {df_all.columns.tolist()}')
df_all.head(3)


In [ ]:
# Filtramos solo lo que nos interesa
df_sector = df_all[df_all['category'] == CATEGORY].reset_index(drop=True).copy()
df_wise   = df_all[df_all['company'] == TARGET].reset_index(drop=True).copy()

n_emp = df_sector['company'].nunique()
print(f'Reseñas del sector "{CATEGORY}": {len(df_sector):,} (de {n_emp} empresas)')
print(f'Reseñas de Wise: {len(df_wise)}')
print()
print('¿Cuántas reseñas tiene Wise de cada puntuación?')
print(df_wise['stars'].value_counts().sort_index().rename({1:'1 estrella',2:'2 estrellas',3:'3 estrellas',4:'4 estrellas',5:'5 estrellas'}))


---
## Paso 3: Limpiar el texto de las reseñas

Los clientes escriben las reseñas de forma libre: emojis, enlaces, mayúsculas, signos...
Antes de que la IA las analice, hay que **estandarizarlas**.

Imagina que tienes 100 formularios escritos a mano con distintas caligrafías.
Antes de procesarlos, los pasas a limpio para que el ordenador los lea sin errores.

**Qué limpiamos:**
- Convertimos todo a minúsculas
- Eliminamos enlaces web
- Eliminamos emojis
- Eliminamos signos de puntuación
- Unimos el título y el texto de la reseña (así la IA tiene más contexto)


In [ ]:
# Miramos algunas reseñas originales antes de limpiarlas
print('Ejemplos de reseñas originales:')
for r in df_wise['review'].sample(3, random_state=42).tolist():
    print('-', r[:150])
    print()


In [ ]:
# Función de limpieza
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()                              # todo a minúsculas
    text = re.sub(r'http\S+|www\.\S+', '', text)    # eliminar enlaces
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)      # eliminar emojis
    text = re.sub(r'[^a-z0-9\s]', ' ', text)        # eliminar puntuación
    text = re.sub(r'\s+', ' ', text).strip()         # espacios múltiples
    return text

def combine_text(row):
    t = str(row['title'])  if pd.notna(row.get('title'))  else ''
    r = str(row['review']) if pd.notna(row.get('review')) else ''
    return clean_text(t + ' ' + r)

# Aplicamos la limpieza
df_wise['text_clean']   = df_wise.apply(combine_text, axis=1)
df_sector['text_clean'] = df_sector.apply(combine_text, axis=1)

print('Texto limpio (mismo ejemplo que antes):')
print(df_wise['text_clean'].iloc[0])


---
## Paso 4: El detector de sentimiento — ¿Está el cliente contento?

Usamos un programa de IA llamado **BERT** (desarrollado por Google) que ha aprendido
a leer texto en 6 idiomas y a decir si el autor estaba contento o descontento.

**¿Cómo funciona?** (sin tecnicismos)

El programa ha leído millones de reseñas de productos y aprendió a asociar palabras
con emociones. Cuando le das una reseña nueva, la puntúa de 1 a 5 estrellas.
Nosotros agrupamos esas estrellas en tres categorías:

- **1-2 estrellas → Cliente descontento (Negativo)**
- **3 estrellas → Cliente indiferente (Neutro)**
- **4-5 estrellas → Cliente satisfecho (Positivo)**

> La primera vez que ejecutes esta celda descargará el modelo (~700 MB).
> Las siguientes veces lo cargará desde tu ordenador, mucho más rápido.


In [ ]:
# Cargamos el detector de sentimiento
# (puede tardar 1-2 minutos la primera vez)
sentiment_pipe = pipeline(
    'text-classification',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    truncation=True,
    max_length=512
)
print('Detector de sentimiento listo.')


In [ ]:
# Función que aplica el detector a una lista de textos
def predict_sentiment(texts, pipe, batch_size=32):
    texts_trunc = [t[:512] if t else 'no review' for t in texts]
    preds = pipe(texts_trunc, batch_size=batch_size)
    stars_list, sent_list = [], []
    for p in preds:
        s = int(p['label'].split()[0])  # convierte '4 stars' en 4
        stars_list.append(s)
        sent_list.append(stars_to_label(s))
    return stars_list, sent_list

# Analizamos las 100 reseñas de Wise
print('Analizando las 100 reseñas de Wise...')
sp, sl = predict_sentiment(df_wise['text_clean'].tolist(), sentiment_pipe)
df_wise['stars_pred'] = sp
df_wise['sentiment']  = sl

print('Resultado:')
print(df_wise['sentiment'].value_counts())


In [ ]:
# Ahora analizamos las ~5.700 reseñas del sector completo
# (puede tardar 10-15 minutos)
print('Analizando el sector completo (~5.700 reseñas)...')
print('Esto puede tardar unos minutos. Es normal.')
sp2, sl2 = predict_sentiment(df_sector['text_clean'].tolist(), sentiment_pipe)
df_sector['stars_pred'] = sp2
df_sector['sentiment']  = sl2
print('Análisis del sector completado.')
print(df_sector['sentiment'].value_counts())


---
## Paso 5: ¿Qué porcentaje de clientes está satisfecho con Wise?

Con el sentimiento calculado, ya podemos responder la primera gran pregunta:
**¿Están contentos los clientes de Wise?**

Y la comparamos con la competencia para saber si es algo del sector o específico de Wise.

### Resultado
- **55% de las reseñas son negativas** — más de la mitad de los clientes que escriben están descontentos
- **Solo el 34% son positivas** — muy por debajo de la media del sector (45.9%)
- Wise ocupa el **puesto 23 de 69 empresas** — en la mitad baja del ranking


In [ ]:
# Calculamos el porcentaje de cada sentimiento en Wise
order = ['Positivo', 'Neutro', 'Negativo']
wise_counts = df_wise['sentiment'].value_counts().reindex(order, fill_value=0)
wise_pct    = (wise_counts / wise_counts.sum() * 100).round(1)

# Gráfico de barras — fácil de leer para cualquier persona
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(order, wise_pct.values,
              color=[PALETTE[o] for o in order], edgecolor='white', linewidth=1.5)
for bar, pct in zip(bars, wise_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('¿Cómo valoran los clientes a Wise?', fontsize=14, fontweight='bold')
ax.set_ylabel('% de reseñas')
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_global.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Negativo: {wise_pct["Negativo"]}% | Neutro: {wise_pct["Neutro"]}% | Positivo: {wise_pct["Positivo"]}%')


In [ ]:
# ¿Cómo está Wise comparada con la competencia?
# Calculamos el % de reseñas positivas de cada empresa
sent_comp = (df_sector.groupby('company')['sentiment']
             .value_counts(normalize=True).mul(100).rename('pct').reset_index())
pos_comp  = sent_comp[sent_comp['sentiment'] == 'Positivo'].sort_values('pct', ascending=True)

# Wise aparece en rojo para que sea fácil de localizar
colors = ['#e74c3c' if c == TARGET else '#3498db' for c in pos_comp['company']]

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(pos_comp['company'], pos_comp['pct'], color=colors)
wise_val = pos_comp[pos_comp['company'] == TARGET]['pct'].values
if len(wise_val):
    ax.axvline(wise_val[0], color='#e74c3c', linestyle='--', linewidth=1.5, label='Wise')
ax.set_title('¿Qué % de clientes están satisfechos en cada empresa?\n(Sector Money & Insurance)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('% de reseñas positivas')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_competencia.png', dpi=150, bbox_inches='tight')
plt.show()

pos_comp_reset = pos_comp.reset_index(drop=True)
wise_idx = pos_comp_reset[pos_comp_reset['company'] == TARGET].index[0] + 1
print(f'Wise ocupa el puesto {wise_idx}/{len(pos_comp_reset)} en satisfacción ({wise_val[0]:.1f}% positivo)')
print(f'Media del sector: {pos_comp_reset["pct"].mean():.1f}%')
print(f'Wise está {pos_comp_reset["pct"].mean() - wise_val[0]:.1f} puntos por debajo de la media')


---
## Paso 6: ¿De qué hablan los clientes en sus reseñas?

Saber que hay un 55% de opiniones negativas es importante, pero no basta.
La pregunta de negocio real es: **¿negativas en qué?**

Para responderla usamos un programa que lee todas las reseñas y las agrupa
por tema automáticamente, sin que nadie le diga cuáles son los temas.

**¿Cómo funciona?** (analogía sencilla)

Imagina que tienes 5.800 post-its con reseñas y los extiendes en una mesa enorme.
El programa los va colocando juntos según de qué hablan: los que hablan de
atención al cliente en un montón, los de transferencias en otro, los de la app en otro...
Al final tienes grupos ("topics") con un nombre que resume de qué trata cada uno.

**¿Por qué entrenamos con el sector y no solo con Wise?**

100 reseñas son pocas para que el programa forme grupos estables.
Entrenamos con las 5.700 del sector y luego miramos en qué grupos caen las reseñas de Wise.
El resultado es mucho más fiable.


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

docs = df_sector['text_clean'].tolist()

# Configuración del agrupador de temas
umap_model    = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', prediction_data=True)
vectorizer    = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=3)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics='auto',
    calculate_probabilities=True,
    verbose=True
)

# El programa lee las 5.700 reseñas y las agrupa
# (puede tardar 5-10 minutos)
print('Agrupando reseñas por tema... esto puede tardar unos minutos.')
topics, probs = topic_model.fit_transform(docs)
df_sector['topic'] = topics

n_t = topic_model.get_topic_info()['Topic'].nunique()
print(f'Temas identificados automáticamente: {n_t}')
topic_model.get_topic_info().head(15)


In [ ]:
# Mostramos las palabras clave de cada tema
# Estas palabras nos ayudan a entender de qué trata cada grupo
topic_info = topic_model.get_topic_info()
print('Palabras clave por tema (para que podamos ponerles nombre):')
for tid in sorted(topic_info['Topic'].values):
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)[:8]]
    print(f'Tema {tid}: {", ".join(words)}')


### Le ponemos nombre a cada tema

El programa nos da números (Tema 0, Tema 2, Tema 6...) y palabras clave.
Nosotros revisamos esas palabras y les ponemos un nombre comprensible.

Por ejemplo:
- Tema con palabras `account, open, verify, id, document` → **Apertura de cuenta**
- Tema con palabras `customer, service, support, help, response` → **Atención al cliente**

> Si vuelves a ejecutar el análisis, los números de tema pueden cambiar ligeramente.
> Revisa las palabras clave y ajusta los nombres si es necesario.


In [ ]:
# Asignamos nombres legibles a los temas
TOPIC_LABELS = {
    -1: 'Sin tema claro (reseña muy personal)',
     0: 'Banca general / Fintech',
     2: 'Proceso fácil y rápido',
     6: 'Atención al cliente',
     7: 'Neobancos (Monzo/Monese)',
     9: 'Criptomonedas (Kraken)',
    13: 'Crédito y préstamos (Zopa)',
    20: 'Banca para empresas (Tide)',
    32: 'Verificación de identidad',
    34: 'Apertura de cuenta',
    35: 'Problemas de acceso / App',
}
df_sector['topic_label'] = df_sector['topic'].map(TOPIC_LABELS).fillna('Otros temas')
print('Nombres asignados a los temas. Listo.')


---
## Paso 7: ¿De qué hablan los clientes de Wise específicamente?

Ahora que tenemos los temas del sector, miramos **en qué temas caen las reseñas de Wise**.

**Resultado:**
- **41 de las 100 reseñas** hablan de banca y fintech en general (transferencias, comisiones, tipos de cambio)
- **55 reseñas son muy personales** y específicas — el programa no las puede agrupar con otras
- **Solo 4 reseñas** caen en temas concretos como atención al cliente o apertura de cuenta

Que la mayoría sean genéricas o muy personales nos dice que los clientes de Wise
hablan de su experiencia de forma muy individual, sin seguir un patrón común.


In [ ]:
# ¿En qué temas caen las reseñas de Wise?
df_wise_topics = df_sector[df_sector['company'] == TARGET].copy()
print('Distribución de temas en las reseñas de Wise:')
print(df_wise_topics[['topic','topic_label']].value_counts().to_string())


In [ ]:
# Gráfico: temas en las reseñas de Wise (excluimos las sin tema claro)
df_w_notout = df_wise_topics[df_wise_topics['topic'] != -1]
wise_topic_counts = df_w_notout['topic_label'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
wise_topic_counts.plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('¿De qué hablan los clientes de Wise?\n(reseñas con tema identificado)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Número de reseñas')
sns.despine()
plt.tight_layout()
plt.savefig('wise_topics.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Paso 8: ¿Los clientes están contentos o descontentos en cada tema?

Este es el cruce más valioso para negocio: combinamos el sentimiento con los temas.
En otras palabras: **¿en qué áreas concretas estamos fallando?**

### Lo que encontramos:
- **Banca general / Fintech** (transferencias, comisiones): 44% positivo, 44% negativo — equilibrado
- **Atención al cliente**: 100% negativo — todos los clientes que escriben sobre este tema están descontentos
- **Proceso fácil**: 100% negativo — (sorprendentemente, las reseñas con esta palabra son quejas)
- **Apertura de cuenta**: 100% negativo — el onboarding es un punto de dolor claro


In [ ]:
# Cruzamos sentimiento con tema para las reseñas de Wise
wise_cross = (df_w_notout.groupby(['topic_label','sentiment'])
              .size().reset_index(name='count'))
pivot_wise = wise_cross.pivot_table(index='topic_label', columns='sentiment',
                                     values='count', fill_value=0)
for col in ['Positivo', 'Neutro', 'Negativo']:
    if col not in pivot_wise.columns:
        pivot_wise[col] = 0
pivot_wise = pivot_wise[['Positivo', 'Neutro', 'Negativo']]
pivot_wise_pct = pivot_wise.div(pivot_wise.sum(axis=1), axis=0).mul(100)

print('Sentimiento por tema en Wise (%): ')
print(pivot_wise_pct.round(1))

fig, ax = plt.subplots(figsize=(9, 5))
pivot_wise_pct.plot(kind='barh', stacked=True, ax=ax,
                    color=[PALETTE['Positivo'], PALETTE['Neutro'], PALETTE['Negativo']],
                    edgecolor='white')
ax.set_title('¿En qué temas están contentos vs descontentos los clientes de Wise?',
             fontsize=13, fontweight='bold')
ax.set_xlabel('% de reseñas')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title='Opinión del cliente', bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig('wise_sentimiento_por_topic.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Paso 9: ¿Dónde estamos mejor y peor que la competencia?

Hasta ahora sabemos cómo está Wise. Pero, ¿es normal tener tantas quejas
sobre atención al cliente en el sector financiero, o es específico de Wise?

Para responderlo calculamos la **diferencia (gap)** entre el porcentaje de
clientes satisfechos en Wise y la media del sector en cada tema.

- **Barra verde = Wise lo hace mejor que la media del sector**
- **Barra roja = Wise lo hace peor que la media del sector**

### Lo que encontramos:
- **Única ventaja**: Banca general / Fintech (+13 puntos sobre la media) — el servicio core de transferencias es percibido mejor que el sector
- **Principales debilidades**: Atención al cliente y Verificación de identidad


In [ ]:
def pct_positive(group):
    return (group['sentiment'] == 'Positivo').mean() * 100

df_sect_notout = df_sector[df_sector['topic'] != -1]
sent_by_tc = (df_sect_notout.groupby(['company','topic_label'])
              .apply(pct_positive).reset_index(name='pct_positive'))

sector_avg = sent_by_tc.groupby('topic_label')['pct_positive'].mean().reset_index(name='sector_avg')
wise_avg   = (sent_by_tc[sent_by_tc['company'] == TARGET][['topic_label','pct_positive']]
              .rename(columns={'pct_positive':'wise_pct'}))
compare = sector_avg.merge(wise_avg, on='topic_label', how='left').fillna(0)
compare['gap'] = compare['wise_pct'] - compare['sector_avg']
compare = compare.sort_values('gap', ascending=True)

print('Diferencia Wise vs sector por tema (puntos porcentuales):')
print(compare[['topic_label','wise_pct','sector_avg','gap']].to_string(index=False))


In [ ]:
# Gráfico de diferencias
colors_gap = ['#2ecc71' if g >= 0 else '#e74c3c' for g in compare['gap']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(compare['topic_label'], compare['gap'], color=colors_gap, edgecolor='white')
ax.axvline(0, color='black', linewidth=1.2)
ax.set_title('¿En qué temas supera Wise a la media del sector?\n(verde = por encima, rojo = por debajo)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Diferencia en puntos porcentuales')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
plt.tight_layout()
plt.savefig('wise_gap_competencia_por_topic.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Paso 10: ¿Qué palabras usan los clientes descontentos?

Una última vista cualitativa: una **nube de palabras** que muestra cuáles son
los términos más repetidos en las reseñas negativas de Wise.

Cuanto más grande aparece una palabra, más veces aparece en las quejas.
Es una forma rápida de identificar patrones que los números no capturan del todo.


In [ ]:
from wordcloud import WordCloud

neg_text = ' '.join(df_wise_topics[df_wise_topics['sentiment'] == 'Negativo']['text_clean'].tolist())
wc = WordCloud(width=800, height=400, background_color='white',
               colormap='Reds', max_words=80).generate(neg_text)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Palabras más frecuentes en las quejas de clientes de Wise',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wise_wordcloud_negativo.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Conclusiones: ¿Qué nos dice el análisis?

### Resumen ejecutivo

| Pregunta | Respuesta |
|---|---|
| ¿Están contentos los clientes de Wise? | **No. 55% de las reseñas son negativas** |
| ¿Cómo está Wise frente a la competencia? | **Puesto 23/69 — 12 puntos por debajo de la media del sector** |
| ¿De qué hablan en las reseñas? | **Banca general/fintech (transferencias, comisiones)** |
| ¿Dónde falla Wise? | **Atención al cliente, apertura de cuenta, proceso de verificación** |
| ¿Tiene alguna ventaja? | **Sí: el servicio de transferencias internacionales se valora mejor que la media** |

---

### Recomendaciones de negocio

**1. Prioridad urgente — Atención al cliente**
Es el tema con 100% de quejas. Los clientes sienten que no obtienen ayuda cuando la necesitan.
Cualquier inversión en soporte (tiempos de respuesta, canales, formación) tendrá impacto inmediato en reseñas.

**2. Simplificar el proceso de alta y verificación**
Apertura de cuenta y verificación de identidad también muestran 100% de insatisfacción.
Reducir fricciones en el onboarding puede convertir frustración en lealtad desde el primer día.

**3. Comunicar mejor durante incidencias**
Las quejas incluyen palabras como "blocked", "frozen", "waiting" — los clientes se quedan sin información
cuando hay un problema. Comunicación proactiva y transparente reduciría la sensación de abandono.

**4. Capitalizar la ventaja en transferencias**
El servicio core de Wise (transferencias internacionales, tipos de cambio) se valora mejor que el sector.
Este es el diferencial real de Wise y merece más protagonismo en marketing y comunicación.


In [ ]:
# Resumen numérico final
print('=' * 60)
print('RESUMEN FINAL — WISE.COM EN TRUSTPILOT')
print('=' * 60)

dist = df_wise_topics['sentiment'].value_counts(normalize=True).mul(100).round(1)
print()
print('SATISFACCIÓN DEL CLIENTE:')
for label in ['Positivo', 'Neutro', 'Negativo']:
    val = dist.get(label, 0)
    bar = '█' * int(val / 5)
    print(f'  {label:10}: {val:4.1f}% {bar}')

pos_comp_r = pos_comp.reset_index(drop=True)
wise_rows = pos_comp_r[pos_comp_r['company'] == TARGET].index
if len(wise_rows):
    pos   = wise_rows[0] + 1
    tot   = len(pos_comp_r)
    pct_w = pos_comp_r.loc[wise_rows[0], 'pct']
    media = pos_comp_r['pct'].mean()
    print(f'\nRANKING EN EL SECTOR:')
    print(f'  Wise: puesto {pos} de {tot} empresas')
    print(f'  Wise: {pct_w:.1f}% de clientes satisfechos')
    print(f'  Sector: {media:.1f}% de media')
    print(f'  Diferencia: -{media - pct_w:.1f} puntos por debajo de la media')

print('\n' + '=' * 60)
